In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/eval_config.yaml
/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/agent.yaml
/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/configs/sampling.yaml
/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/prompts/system.md
/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/prompts/analyzer.md
/kaggle/input/datasets/aracelifradejasmunoz/tailorcode-submission-package/sub_agents/code_analyzer.yaml
/kaggle/input/competitions/gemma-4-developer-agent/HARNESS_README.md
/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/eval_config.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/agent.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/configs/sampling.yaml
/kaggle/input/competitions/gemma-4-developer-agen

# tAilorCode - First Evaluation

This notebook is the first private evaluation environment for **tAilorCode**, my submission project for the Google - The Gemma 4 Developer Agent Competition.

The purpose of this first step is to confirm that the competition dataset is available and that the Kaggle environment can access the files required by the evaluation harness.

We are not running the full benchmark yet. First, we verify the environment and inspect the available inputs without modifying the competition data.

In [13]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("Available inputs:")
for item in sorted(input_root.iterdir()):
    print(f"- {item}")

print("\nRelevant files:")
for path in sorted(input_root.rglob("*")):
    if path.name in {"tasks.jsonl", "HARNESS_README.md"}:
        print(path)

Available inputs:
- /kaggle/input/competitions
- /kaggle/input/datasets

Relevant files:
/kaggle/input/competitions/gemma-4-developer-agent/HARNESS_README.md
/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl


## Interpretation

If `tasks.jsonl`, `HARNESS_README.md`, and the competition directories are available, the Kaggle environment is ready for the next step: validating the tAilorCode submission structure and preparing the first focused evaluation task, `fastapi_14786`.

This notebook is private development documentation. It is separate from the official `submission.zip` archive.

## Environment check

The Kaggle environment can access the competition dataset successfully.

The main files detected are:

- `HARNESS_README.md`, which documents the evaluation harness.
- `tasks.jsonl`, which contains the public development tasks.
- The `gemma-4-developer-agent` competition directory, which contains the remaining repository snapshots, graphs, embeddings, and offline dependencies.

The next step is to inspect the dataset structure and validate the tAilorCode submission package.

In [14]:
from pathlib import Path

dataset_root = Path(
    "/kaggle/input/competitions/gemma-4-developer-agent"
)

print("Competition dataset structure:")

for item in sorted(dataset_root.iterdir()):
    item_type = "directory" if item.is_dir() else "file"
    print(f"- [{item_type}] {item.name}")

Competition dataset structure:
- [file] HARNESS_README.md
- [directory] docker
- [directory] embeddings
- [directory] graphs
- [directory] sample_submission
- [directory] sandbox
- [directory] snapshots
- [file] tasks.jsonl
- [directory] wheels


## Create the tAilorCode submission

The declarative tAilorCode files are embedded in this notebook. This avoids Internet access and avoids relying on additional Kaggle input datasets during the hidden competition rerun.

In [2]:
from pathlib import Path
import shutil

kaggle_working = Path("/kaggle/working")
working_root = kaggle_working if kaggle_working.exists() else Path.cwd() / "kaggle-working"
repository_path = working_root / "tailorcode-gemma-developer-agent"
submission_path = repository_path / "submission"

if repository_path.exists():
    shutil.rmtree(repository_path)

embedded_files = {
    "agent.yaml": """name: tAilorCode
model: gemma-4-31b-it-qat-w4a16-ct
description: A grounded autonomous coding agent for verified Python repository fixes.
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
  - agent_tool:
      config_path: sub_agents/code_analyzer.yaml
      skip_summarization: true
generate_content_config: !include configs/sampling.yaml
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 300
  max_tool_calls: 50
  max_time_minutes: 30
  max_turns: 100
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "prompts/system.md": """You are tAilorCode, an autonomous software engineering agent working inside /workspace.

Resolve the supplied Python repository issue with the smallest source-code patch that passes verification tests.

Workflow:
1. Read the issue and identify acceptance criteria.
2. Inspect the repository and relevant source and test files.
3. Form an evidence-based root-cause hypothesis before editing.
4. Use code-intelligence tools with concrete symbol names when available.
5. Make a minimal source-only change. Never modify tests or test configuration.
6. Run targeted tests or focused assertions. Do not run a bare full-repository test command.
7. Remove temporary files and review the final diff.
8. Call submit_patch() only after verification. It must be your final tool call.

The environment is offline. Do not install packages or download files. Work only under /workspace and optimize for a correct, reviewable, reproducible patch.
""",
    "prompts/analyzer.md": """You are tAilorCode's read-only repository analyst.

Use read_file, search_similar_code, get_code_neighbors, and get_code_subgraph to locate the likely root cause. Do not modify files or run commands.

Return a concise report with relevant paths and symbols, evidence, root cause, and the smallest implementation direction for the main agent.
""",
    "sub_agents/code_analyzer.yaml": """name: tAilorCode_analyzer
description: A read-only specialist that locates relevant files, symbols, and root causes.
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include ../prompts/analyzer.md
tools:
  - read_file
  - search_similar_code
  - get_code_neighbors
  - get_code_subgraph
generate_content_config: !include ../configs/sampling.yaml
""",
}

for relative_path, content in embedded_files.items():
    target_path = submission_path / relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(content)

print("tAilorCode created inside the notebook workspace.")
print("Submission path:", submission_path)
for path in sorted(submission_path.rglob("*")):
    if path.is_file():
        print(path.relative_to(repository_path))

tAilorCode created inside the notebook workspace.
Submission path: /Users/LDAAFM/Downloads/kaggle-working/tailorcode-gemma-developer-agent/submission
submission/agent.yaml
submission/configs/sampling.yaml
submission/eval_config.yaml
submission/prompts/analyzer.md
submission/prompts/system.md
submission/sub_agents/code_analyzer.yaml


## Validate the tAilorCode submission

The repository has been cloned successfully. This cell checks that the submission contains the required root configuration, prompts, sub-agent configuration, and the correct competition model declaration.

In [3]:
from pathlib import Path

kaggle_working = Path("/kaggle/working")
working_root = kaggle_working if kaggle_working.exists() else Path.cwd() / "kaggle-working"
submission_path = working_root / "tailorcode-gemma-developer-agent" / "submission"

required_files = [
    "agent.yaml",
    "eval_config.yaml",
    "configs/sampling.yaml",
    "prompts/system.md",
    "prompts/analyzer.md",
    "sub_agents/code_analyzer.yaml",
]

for relative_path in required_files:
    file_path = submission_path / relative_path
    assert file_path.is_file(), f"Missing file: {relative_path}"
    print(f"OK: {relative_path}")

agent_config = (submission_path / "agent.yaml").read_text()
analyzer_config = (
    submission_path / "sub_agents" / "code_analyzer.yaml"
).read_text()

expected_model = "gemma-4-31b-it-qat-w4a16-ct"

assert expected_model in agent_config
assert expected_model in analyzer_config
assert "submit_patch" in agent_config
assert "agent_tool:" in agent_config

print("\nThe tAilorCode submission structure is valid.")
print("The required Gemma 4 model is declared correctly.")

OK: agent.yaml
OK: eval_config.yaml
OK: configs/sampling.yaml
OK: prompts/system.md
OK: prompts/analyzer.md
OK: sub_agents/code_analyzer.yaml

The tAilorCode submission structure is valid.
The required Gemma 4 model is declared correctly.


## Check the evaluation harness

The tAilorCode submission is structurally valid. The next check determines whether the Kaggle environment already provides the `swegemma` evaluation command and its supporting Python packages.

In [17]:
import importlib.util
import shutil

checks = {
    "swegemma command": shutil.which("swegemma"),
    "swegemma Python package": importlib.util.find_spec("swegemma"),
    "Google ADK package": importlib.util.find_spec("google.adk"),
    "Docker command": shutil.which("docker"),
}

for name, result in checks.items():
    status = "available" if result else "not available"
    print(f"{name}: {status}")

swegemma command: not available
swegemma Python package: not available
Google ADK package: available
Docker command: not available


## Package the tAilorCode submission

The submission configuration is valid. Because the standard Kaggle Notebook does not provide the local `swegemma` CLI or Docker runtime, we will now package the declarative agent configuration in the required `submission.zip` format.

The archive must contain `agent.yaml` at its root.

In [4]:
from pathlib import Path
import shutil
import zipfile

kaggle_working = Path("/kaggle/working")
working_root = kaggle_working if kaggle_working.exists() else Path.cwd() / "kaggle-working"
repository_path = working_root / "tailorcode-gemma-developer-agent"
submission_path = repository_path / "submission"
archive_base = working_root / "submission"

archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=submission_path,
    )
)

print("Created:", archive_path)
print("Size:", archive_path.stat().st_size, "bytes")

with zipfile.ZipFile(archive_path) as archive:
    print("\nArchive contents:")
    for name in archive.namelist():
        print("-", name)

    assert "agent.yaml" in archive.namelist()
    assert "eval_config.yaml" in archive.namelist()

print("\nThe submission archive is ready.")

Created: /Users/LDAAFM/Downloads/kaggle-working/submission.zip
Size: 2447 bytes

Archive contents:
- configs/
- prompts/
- sub_agents/
- agent.yaml
- eval_config.yaml
- configs/sampling.yaml
- prompts/system.md
- prompts/analyzer.md
- sub_agents/code_analyzer.yaml

The submission archive is ready.
